#### Data Wrangling — Kualitas Udara & Dampak Kesehatan
Alur: **Gathering → Assessing → Cleaning → Integrasi → Simpan**

#### Setup

In [5]:
%pip install pandas numpy matplotlib seaborn

  Using cached pandas-3.1.0-cp312-cp312-win_amd64.whl.metadata (18 kB)
  Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached matplotlib-3.11.2-cp312-cp312-win_amd64.whl.metadata (80 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached contourpy-1.4.0-cp312-cp312-win_amd64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.68.0-cp312-cp312-win_amd64.whl.metadata (142 kB)
  Using cached kiwisolver-1.5.1-cp312-cp312-win_amd64.whl.metadata (5.2 kB)
Using cached pandas-3.1.0-cp312-cp312-win_amd64.whl (10.7 MB)
Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl (12.6 MB)
Using cached matplotlib-3.11.2-cp312-cp312-win_amd64.whl (9.3 MB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Using cached contourpy-1.4.0-cp312-cp312-win_amd64.whl (234 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.68.0-cp312-cp312-win_amd64.whl (2.5 MB)
U

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'c:\\Users\\Mega Agustina\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\numpy\\_core\\tests\\test_records.py'
Consider using the `--user` option or check the permissions.


[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
import pandas as pd, numpy as np, glob, os
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

DATA = "../data"
POLLUTANTS = ["pm25_ug_m3", "pm10_ug_m3", "so2_ug_m3", "co_ug_m3", "o3_ug_m3", "no2_ug_m3"]
METEO = ["temperature_c", "humidity_pct", "wind_speed_ms", "wind_direction_deg"]
NUM_COLS = POLLUTANTS + METEO
HEALTH_COUNTS = ["resp_admissions_under5", "resp_admissions_over60", "asthma_emergency_cases",
                 "copd_admissions", "resp", "cvd", "death"]
HEALTH_WEATHER = ["temp_c", "dptp_c", "rhum_pct"]

EXPECTED_ROWS = 24432            # 1 Jan 2024 00:00 s.d. 14 Okt 2026 23:00, per jam (dari metadata_dates)

# Kode penanda "data hilang" yang terlihat di data (-99.9 pada angin, -999 dan 9999 pada arah angin)
SENTINELS = [-9999, -999, -99.9, 9999, 99999]

# ⚙️ SESUAIKAN: batas fisik / masuk akal. Nilai di luar batas dianggap error sensor → NaN.
LIMITS = {
    "pm25_ug_m3": (0, 1000), "pm10_ug_m3": (0, 2000), "so2_ug_m3": (0, 2000),
    "co_ug_m3": (0, 100000), "o3_ug_m3": (0, 1000), "no2_ug_m3": (0, 1000),
    "temperature_c": (-10, 50), "humidity_pct": (0, 100),
    "wind_speed_ms": (0, 60), "wind_direction_deg": (0, 360),
}

cleaning_log = []
def log(step, n, note=""):
    cleaning_log.append({"step": step, "rows_affected": int(n), "note": note})
    print(f"[LOG] {step}: {int(n):,} {note}")

def city_key(s):
    """Kunci join wilayah. Awalan Kota/Kabupaten TIDAK dihapus (Kota Bogor != Kabupaten Bogor)."""
    return (s.astype(str).str.replace("Kota Administrasi ", "", regex=False)
             .str.strip().str.replace(r"\s+", " ", regex=True).str.lower())

CANDIDATE_FORMATS = [
    "%Y-%m-%d %H:%M:%S.%f", "%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M",
    "%Y-%m-%dT%H:%M:%S.%f", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%dT%H:%M",
    "%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M", "%d-%m-%Y %H:%M:%S", "%d-%m-%Y %H:%M",
    "%m/%d/%Y %H:%M:%S", "%m/%d/%Y %H:%M",
    "%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y", "%m/%d/%Y",
]

def parse_ts(s):
    """Baca timestamp per file. Mencoba epoch lalu beberapa format; pilih yang gagal-bacanya paling sedikit.
    Return: (series datetime, nama format, persen gagal)."""
    s = s.astype("string").str.strip()
    ok = s.notna()
    if ok.sum() == 0:
        return pd.to_datetime(s, errors="coerce"), "kosong", 100.0
    if s[ok].str.fullmatch(r"\d{9,13}").mean() > 0.9:                 # epoch detik / milidetik
        v = pd.to_numeric(s, errors="coerce")
        unit = "ms" if v.dropna().median() > 1e11 else "s"
        out = pd.to_datetime(v, unit=unit, errors="coerce")
        return out, f"epoch_{unit}", round(out[ok].isna().mean() * 100, 3)
    best = (None, None, 1.0)
    for fmt in CANDIDATE_FORMATS:
        out = pd.to_datetime(s, format=fmt, errors="coerce")
        fail = out[ok].isna().mean()
        if fail < best[2]:
            best = (out, fmt, fail)
        if fail == 0:
            break
    return best[0], best[1], round(best[2] * 100, 3)

In [24]:
# Baca tiap file AQMS sebagai TEKS (agar kolom campur angka+teks tidak error), lalu:
#  - file berkolom *_ppm diseragamkan namanya ke *_ug_m3 dan ditandai unit_raw="ppm"
#  - timestamp dibaca PER FILE (format antar file berbeda)
PPM_RENAME = {f"{p}_ppm": f"{p}_ug_m3" for p in ["pm25", "pm10", "so2", "co", "o3", "no2"]}

files = sorted(glob.glob(f"{DATA}/aqms/*_AQMS.csv"))
print(len(files), "file AQMS ditemukan")

frames, info = [], []
for f in files:
    name = os.path.basename(f)
    df = pd.read_csv(f, dtype=str)
    is_ppm = ("pm25_ppm" in df.columns) and ("pm25_ug_m3" not in df.columns)
    if is_ppm:
        df = df.rename(columns=PPM_RENAME)
    df["unit_raw"] = "ppm" if is_ppm else "ug/m3"
    df["source_file"] = name
    df["ts_start"], fmt, fail = parse_ts(df["timestamp_start"])
    df["ts_end"], _, _ = parse_ts(df["timestamp_end"])
    info.append({"file": name, "baris": len(df), "selisih_vs_harapan": len(df) - EXPECTED_ROWS,
                 "unit": df["unit_raw"].iloc[0], "format_ts": fmt, "gagal_baca_ts_%": fail,
                 "contoh_ts_mentah": df["timestamp_start"].iloc[0]})
    if fail > 1:
        print(f"⚠️ {name}: format timestamp tidak dikenali ({fail}% gagal). Contoh mentah:", df["timestamp_start"].head(3).tolist())
    frames.append(df)

aqms_raw = pd.concat(frames, ignore_index=True)
del frames
file_info = pd.DataFrame(info).set_index("file")
print("Total baris AQMS:", f"{len(aqms_raw):,}", "| kolom:", aqms_raw.shape[1])
display(file_info)

27 file AQMS ditemukan
Total baris AQMS: 667,981 | kolom: 19


,baris,selisih_vs_harapan,unit,format_ts,gagal_baca_ts_%,contoh_ts_mentah
file,,,,,,
BGL-01_Bengkulu_AQMS.csv,24432,0,ug/m3,%Y-%m-%d %H:%M:%S.%f,0.000,2024-01-01 00:02:10.000
BGR-01_Cisarua_AQMS.csv,24432,0,ug/m3,%Y-%m-%d %H:%M:%S,0.000,2024-01-01 00:00:00
BGR-02_Baranangsiang_AQMS.csv,24432,0,ug/m3,%Y-%m-%d %H:%M:%S,0.000,2024-01-01 00:00:00
BJB-01_Banjarbaru_AQMS.csv,24432,0,ppm,%Y-%m-%d %H:%M:%S.%f,0.000,2024-01-01 00:01:15.420
BTM-01_Batam_AQMS.csv,24432,0,ug/m3,%Y-%m-%d %H:%M:%S.%f,0.000,2023-12-31 23:48:15.100
IDP-01_Indrapuri_AQMS.csv,24432,0,ug/m3,%Y-%m-%d %H:%M:%S,0.000,2024-01-01 00:00:00
JKT-01_Gambir_AQMS.csv,24432,0,ug/m3,%Y-%m-%d %H:%M:%S,0.000,2024-01-01 00:00:00
JKT-02_Marunda_AQMS.csv,24432,0,ug/m3,%Y-%m-%d %H:%M:%S,0.000,2024-01-01 00:00:00
JKT-03_Cengkareng_AQMS.csv,24432,0,ug/m3,%Y-%m-%d %H:%M:%S,0.000,2024-01-01 00:00:00


## 2. Assessing — cek data kosong, duplikat, dan tidak sesuai
### 2.1 Tabel master & metadata

In [25]:
tables = {"sites": (sites, "location_id"), "sensors": (sensors, "sensor_id"),
          "hospitals": (hospitals, "hospital_id"), "m_dates": (m_dates, "location_id")}
rows = []
for name, (df, key) in tables.items():
    rows.append({"tabel": name, "baris": len(df), "kolom": df.shape[1],
                 "kolom_ada_kosong": int((df.isna().sum() > 0).sum()),
                 "duplikat_baris": int(df.duplicated().sum()),
                 "duplikat_kunci": int(df[key].duplicated().sum())})
display(pd.DataFrame(rows))

unit_cols = [c for c in sites.columns if c.endswith("_unit")]
print("Lokasi dengan satuan bukan ug/m3 (menurut sites.csv):")
display(sites.loc[(sites[unit_cols] != "ug/m3").any(axis=1), ["location_id", "kota_kabupaten"] + unit_cols])

sensors["tanggal_instalasi"] = pd.to_datetime(sensors["tanggal_instalasi"], errors="coerce")
print("Sensor tidak Active:")
display(sensors.loc[sensors["status_operasional"] != "Active", ["sensor_id", "location_id", "nama_sensor", "status_operasional"]])

print("Nama kota cocok TANPA normalisasi:", len(set(sites["kota_kabupaten"]) & set(hospitals["kota_kabupaten"])), "dari", len(sites))
print("Nama kota cocok SETELAH normalisasi:", len(set(city_key(sites["kota_kabupaten"])) & set(city_key(hospitals["kota_kabupaten"]))), "dari", len(sites))

days = pd.date_range(START.normalize(), END.normalize())
per_year = pd.Series(days.year).value_counts().sort_index()
need = np.ceil(0.8 * np.where(per_year.index % 4 == 0, 366, 365)).astype(int)
iku_ready = pd.DataFrame({"hari_tersedia": per_year, "syarat_80_persen": need})
iku_ready["bisa_lolos_syarat_IKU"] = iku_ready["hari_tersedia"] >= iku_ready["syarat_80_persen"]
display(iku_ready)

,tabel,baris,kolom,kolom_ada_kosong,duplikat_baris,duplikat_kunci
0,sites,27,27,0,0,0
1,sensors,27,10,0,0,0
2,hospitals,27,7,0,0,0
3,m_dates,27,6,0,0,0


Lokasi dengan satuan bukan ug/m3 (menurut sites.csv):


,location_id,kota_kabupaten,pm25_unit,pm10_unit,so2_unit,co_unit,o3_unit,no2_unit
21,LOC-PSW-01,Kabupaten Pesawaran,ppm,ppm,ppm,ppm,ppm,ppm
23,LOC-MDN-01,Kota Medan,ppm,ppm,ppm,ppm,ppm,ppm
25,LOC-BJB-01,Kota Banjarbaru,ppm,ppm,ppm,ppm,ppm,ppm


Sensor tidak Active:


,sensor_id,location_id,nama_sensor,status_operasional
20,SENS-021,LOC-PLB-01,AQMS BMKG Talang Betutu Palembang,Intermittent
22,SENS-023,LOC-MER-01,AQMS BMKG Mopah Merauke,Intermittent
23,SENS-024,LOC-MDN-01,AQMS BMKG Stasiun Klimatologi Medan,Intermittent
26,SENS-027,LOC-JMB-02,AQMS BMKG Muaro Jambi,Defective


Nama kota cocok TANPA normalisasi: 24 dari 27
Nama kota cocok SETELAH normalisasi: 27 dari 27


,hari_tersedia,syarat_80_persen,bisa_lolos_syarat_IKU
2024,366,293,True
2025,365,292,True
2026,287,292,False


### 2.2 AQMS — data kosong

In [26]:
cols = ["ts_start", "ts_end"] + NUM_COLS + ["qc_flag"]
miss = aqms_raw[cols].isna().sum().to_frame("jumlah_kosong")
miss["persen"] = miss["jumlah_kosong"] / len(aqms_raw) * 100
display(miss)

print("Persen kosong per file (urut terbesar, 8 teratas):")
per_file_miss = aqms_raw.groupby("source_file")[NUM_COLS].apply(lambda d: d.isna().mean() * 100)
display(per_file_miss.assign(maks=per_file_miss.max(axis=1)).sort_values("maks", ascending=False).head(8).round(1))

,jumlah_kosong,persen
ts_start,0,0.000
ts_end,0,0.000
pm25_ug_m3,20932,3.134
pm10_ug_m3,20942,3.135
so2_ug_m3,20164,3.019
co_ug_m3,20170,3.020
o3_ug_m3,20171,3.020
no2_ug_m3,20177,3.021
temperature_c,18632,2.789
humidity_pct,18632,2.789


Persen kosong per file (urut terbesar, 8 teratas):


,pm25_ug_m3,pm10_ug_m3,so2_ug_m3,co_ug_m3,o3_ug_m3,no2_ug_m3,temperature_c,humidity_pct,wind_speed_ms,wind_direction_deg,maks
source_file,,,,,,,,,,,
MER-01_Merauke_AQMS.csv,40.000,40.000,40.000,40.000,40.000,40.000,40.000,40.000,40.000,40.000,40.000
JMB-02_MuaroJambi_AQMS.csv,19.100,19.100,18.900,19.000,19.000,19.000,15.200,15.200,15.200,0.000,19.100
PLB-01_TalangBetutu_AQMS.csv,14.100,14.100,14.100,14.100,14.100,14.100,14.100,14.100,14.100,14.100,14.100
SM2-01_Samarinda_AQMS.csv,3.000,3.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,3.000
MDN-01_Medan_AQMS.csv,3.000,3.000,3.000,3.000,3.000,3.000,0.000,0.000,0.000,0.000,3.000
IDP-01_Indrapuri_AQMS.csv,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
BTM-01_Batam_AQMS.csv,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
BGR-02_Baranangsiang_AQMS.csv,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000


### 2.3 AQMS — data duplikat & jumlah baris

In [27]:
jam = aqms_raw["ts_start"].dt.round("h")
print("measurement_id ganda               :", int(aqms_raw["measurement_id"].duplicated().sum()))
print("baris identik penuh                :", int(aqms_raw.duplicated().sum()))
print("sensor_id + timestamp mentah ganda :", int(aqms_raw.duplicated(["sensor_id", "timestamp_start"]).sum()))
print("sensor_id + jam (dibulatkan) ganda :", int(pd.DataFrame({"s": aqms_raw["sensor_id"], "j": jam}).dropna().duplicated().sum()))

per = (aqms_raw.assign(jam=jam).groupby("source_file")
       .agg(baris=("jam", "size"), jam_unik=("jam", "nunique")))
per["selisih_vs_harapan"] = per["baris"] - EXPECTED_ROWS
per["jam_hilang"] = EXPECTED_ROWS - per["jam_unik"]
print("\nFile yang jumlah barisnya tidak sama dengan harapan, atau punya jam hilang:")
display(per[(per["selisih_vs_harapan"] != 0) | (per["jam_hilang"] > 0)])

besar = per["selisih_vs_harapan"].idxmax()
sel = aqms_raw.loc[aqms_raw["source_file"] == besar, "ts_start"].dropna().sort_values()
print(f"\nJarak antar pengukuran di {besar} (kelebihan baris terbesar):")
display(sel.diff().value_counts().head(5).to_frame("jumlah"))

measurement_id ganda               : 0
baris identik penuh                : 0
sensor_id + timestamp mentah ganda : 223
sensor_id + jam (dibulatkan) ganda : 12172

File yang jumlah barisnya tidak sama dengan harapan, atau punya jam hilang:


,baris,jam_unik,selisih_vs_harapan,jam_hilang
source_file,,,,
JMB-02_MuaroJambi_AQMS.csv,24432,24209,0,223
MDN-01_Medan_AQMS.csv,20800,20800,-3632,3632
PLB-01_TalangBetutu_AQMS.csv,36381,24432,11949,0



Jarak antar pengukuran di PLB-01_TalangBetutu_AQMS.csv (kelebihan baris terbesar):


,jumlah
ts_start,
0 days 01:00:00,20448
0 days 00:15:00,15932


### 2.4 AQMS — data tidak sesuai (teks error, kode penanda, di luar batas, timestamp, satuan, QC)

In [28]:
# (a) Teks di kolom yang seharusnya angka
def text_report(df, cols):
    rows = []
    for c in cols:
        conv = pd.to_numeric(df[c], errors="coerce")
        bad = df.loc[conv.isna() & df[c].notna()]
        if len(bad):
            top = bad[c].value_counts().head(5)
            rows.append({"kolom": c, "jumlah_teks": len(bad),
                         "contoh": ", ".join(f"{k} ({v})" for k, v in top.items()),
                         "file_asal": ", ".join(bad["source_file"].value_counts().head(3).index)})
    return pd.DataFrame(rows)

display(text_report(aqms_raw, NUM_COLS))

,kolom,jumlah_teks,contoh,file_asal
0,pm25_ug_m3,6290,"0xDEADBEEF (970), ERR_COMM_TIMEOUT (942), #VAL...",JMB-02_MuaroJambi_AQMS.csv
1,pm10_ug_m3,6283,"0xDEADBEEF (967), ERR_COMM_TIMEOUT (936), #VAL...",JMB-02_MuaroJambi_AQMS.csv
2,so2_ug_m3,6286,"ERR_COMM_TIMEOUT (924), 0xDEADBEEF (913), TAPE...",JMB-02_MuaroJambi_AQMS.csv
3,co_ug_m3,6407,"0xDEADBEEF (977), ERR_COMM_TIMEOUT (936), #VAL...",JMB-02_MuaroJambi_AQMS.csv
4,o3_ug_m3,6321,"0xDEADBEEF (978), VOLT_SURGE_99V (871), BAD_CH...",JMB-02_MuaroJambi_AQMS.csv
5,no2_ug_m3,6279,"ERR_COMM_TIMEOUT (966), 0xDEADBEEF (901), BAD_...",JMB-02_MuaroJambi_AQMS.csv
6,temperature_c,124,ERR (124),JMB-02_MuaroJambi_AQMS.csv
7,humidity_pct,149,ERR (149),JMB-02_MuaroJambi_AQMS.csv


In [29]:
# (b) Statistik angka setelah teks dibuang  &  (c) pelanggaran batas
num = aqms_raw[NUM_COLS].apply(pd.to_numeric, errors="coerce")
display(num.describe(percentiles=[.01, .5, .99]).T)

rows = {}
for c, (lo, hi) in LIMITS.items():
    rows[c] = {"di_bawah_min": int((num[c] < lo).sum()), "di_atas_max": int((num[c] > hi).sum())}
display(pd.DataFrame(rows).T)

print("Nilai di luar batas yang paling sering muncul (pola berulang = kode penanda / sentinel):")
for c, (lo, hi) in LIMITS.items():
    out = num.loc[(num[c] < lo) | (num[c] > hi), c]
    if len(out):
        print(f"  {c}: {out.value_counts().head(4).to_dict()}")

c:\Users\Mega Agustina\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\nanops.py:1324: RuntimeWarning: invalid value encountered in subtract
  sqr = (avg - values) ** 2
c:\Users\Mega Agustina\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\nanops.py:1324: RuntimeWarning: invalid value encountered in subtract
  sqr = (avg - values) ** 2
c:\Users\Mega Agustina\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\nanops.py:1324: RuntimeWarning: invalid value encountered in subtract
  sqr = (avg - values) ** 2
c:\Users\Mega Agustina\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\nanops.py:1324: RuntimeWarning: invalid value encountered in subtract
  sqr = (avg - values) ** 2
c:\Users\Mega Agustina\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\nanops.py:1324: RuntimeWarning: invalid value encountered in subtract
  sqr = (avg - values) ** 2
c:\Users\Mega Agustina\AppData\Local\Programs

,count,mean,std,min,1%,50%,99%,max
pm25_ug_m3,"640,759.000",inf,NaN,"-9,999.000",-259.196,25.000,999.000,inf
pm10_ug_m3,"640,756.000",inf,NaN,"-9,999.000",-516.726,35.600,359.204,inf
so2_ug_m3,"641,531.000",inf,NaN,"-9,999.000",-99.000,7.900,400.000,inf
co_ug_m3,"641,404.000",inf,NaN,"-9,999.000",-999.000,370.300,"4,500.000",inf
o3_ug_m3,"641,489.000",inf,NaN,"-9,999.000",-50.000,12.300,82.800,inf
no2_ug_m3,"641,525.000",inf,NaN,"-9,999.000",-40.000,18.900,82.650,inf
temperature_c,"649,225.000",26.874,173.251,-999.000,-999.000,25.700,850.000,"1,450.500"
humidity_pct,"649,200.000",97.360,125.051,-150.000,58.900,82.900,999.900,999.900
wind_speed_ms,"649,349.000",2.391,34.089,-99.900,-99.900,2.100,5.200,350.000
wind_direction_deg,"653,067.000",271.207,"1,038.080",-999.000,-999.000,176.000,"9,999.000","9,999.000"


,di_bawah_min,di_atas_max
pm25_ug_m3,9369,6368
pm10_ug_m3,7582,874
so2_ug_m3,9449,5729
co_ug_m3,7571,799
o3_ug_m3,7627,5729
no2_ug_m3,7660,5732
temperature_c,10856,10361
humidity_pct,2392,15863
wind_speed_ms,15597,4901
wind_direction_deg,12192,12024


Nilai di luar batas yang paling sering muncul (pola berulang = kode penanda / sentinel):
  pm25_ug_m3: {-9999.0: 891, inf: 873, -0.55: 20, -2.12: 20}
  pm10_ug_m3: {-9999.0: 889, inf: 873, -1284.6: 3, -440.03: 3}
  so2_ug_m3: {-99.0: 6693, 50000.0: 4901, -9999.0: 976, inf: 828}
  co_ug_m3: {-999.0: 6693, -9999.0: 878, inf: 799}
  o3_ug_m3: {-50.0: 6693, 9999.0: 4901, -9999.0: 934, inf: 828}
  no2_ug_m3: {-40.0: 6693, 8888.0: 4901, -9999.0: 967, inf: 831}
  temperature_c: {-999.0: 8474, 850.0: 4901, 1450.5: 2375, -273.2: 2249}
  humidity_pct: {999.9: 10962, 500.0: 4901, -150.0: 2208, -50.0: 184}
  wind_speed_ms: {-99.9: 8474, -99.0: 7123, 350.0: 4901}
  wind_direction_deg: {-999.0: 8474, 9999.0: 7123, 1440.0: 4901, -1.0: 3718}


In [30]:
# (d) Timestamp & logika
dur = aqms_raw["ts_end"] - aqms_raw["ts_start"]
ts_out = (aqms_raw["ts_start"].isna() | (aqms_raw["ts_start"] < START - pd.Timedelta(hours=1))
          | (aqms_raw["ts_start"] > END + pd.Timedelta(hours=1)))
off_hour = (aqms_raw["ts_start"].dt.minute != 0) | (aqms_raw["ts_start"].dt.second != 0)
checks = pd.Series({
    "ts_start gagal dibaca / di luar rentang metadata": int(ts_out.sum()),
    "ts_end <= ts_start": int((dur <= pd.Timedelta(0)).sum()),
    "durasi pengukuran bukan 1 jam": int((dur.notna() & (dur != pd.Timedelta(hours=1))).sum()),
    "ts_start tidak tepat jam (menit/detik != 0)": int(off_hour.sum()),
    "PM2.5 > PM10 (toleransi 5%)": int((num["pm25_ug_m3"] > num["pm10_ug_m3"] * 1.05).sum()),
}, name="jumlah_baris")
display(checks.to_frame())

print("File dengan timestamp di luar rentang:")
display(aqms_raw.loc[ts_out].groupby("source_file").size().to_frame("baris"))

,jumlah_baris
ts_start gagal dibaca / di luar rentang metadata,132
ts_end <= ts_start,92
durasi pengukuran bukan 1 jam,16025
ts_start tidak tepat jam (menit/detik != 0),399097
PM2.5 > PM10 (toleransi 5%),14401


File dengan timestamp di luar rentang:


,baris
source_file,
JMB-02_MuaroJambi_AQMS.csv,132


In [10]:
sites     = pd.read_csv(f"{DATA}/master/sites.csv")
sensors   = pd.read_csv(f"{DATA}/master/sensors.csv")
hospitals = pd.read_csv(f"{DATA}/master/hospitals.csv")
health    = pd.read_csv(f"{DATA}/health/HLT-01_health_outcomes_daily.csv")
m_dates   = pd.read_csv(f"{DATA}/metadata/metadata_dates.csv")
m_sensors = pd.read_csv(f"{DATA}/metadata/metadata_sensors.csv")
...

Ellipsis

In [31]:
# (e) qc_flag: bandingkan dengan yang tercatat di metadata
DOC_FLAGS = {"VALID", "SUSPECT", "CALIBRATING", "MISSING"}
flags = aqms_raw["qc_flag"].value_counts(dropna=False)
display(flags.to_frame("jumlah").assign(persen=lambda d: d["jumlah"] / d["jumlah"].sum() * 100))
print("Flag yang TIDAK terdokumentasi di metadata_sensors:", [f for f in flags.index if f not in DOC_FLAGS])

# (f) Satuan: bandingkan skala stasiun ppm dengan stasiun lain
is_ppm = aqms_raw["unit_raw"].eq("ppm")
MW = {"so2_ug_m3": 64.07, "no2_ug_m3": 46.01, "co_ug_m3": 28.01, "o3_ug_m3": 48.00}
scale = pd.DataFrame({"median_stasiun_ppm": num.loc[is_ppm, POLLUTANTS].median(),
                      "median_stasiun_ug": num.loc[~is_ppm, POLLUTANTS].median()})
scale["rasio_ug_per_ppm"] = scale["median_stasiun_ug"] / scale["median_stasiun_ppm"]
scale["faktor_fisik_gas"] = [MW[c] * 1000 / 24.45 if c in MW else np.nan for c in scale.index]
print("Stasiun ppm:", sorted(aqms_raw.loc[is_ppm, "source_file"].unique()))
display(scale)
print("Jika rasio ≈ faktor_fisik_gas → nilainya memang ppm (konversi benar). Jika rasio ≈ 1 → label ppm keliru.")
print("PM2.5/PM10 tidak punya faktor fisik: lihat rasio untuk memutuskan PM_PPM_FACTOR di bagian 3.2.")

,jumlah,persen
qc_flag,,
VALID,599185,89.701
SUSPECT,19741,2.955
MISSING,13675,2.047
CALIBRATING,8072,1.208
WARMUP,3764,0.563
PARSER_FAIL,3633,0.544
FATAL_ERR,3546,0.531
GARBAGE,3537,0.530
COMM_LOST,3433,0.514


Flag yang TIDAK terdokumentasi di metadata_sensors: ['WARMUP', 'PARSER_FAIL', 'FATAL_ERR', 'GARBAGE', 'COMM_LOST', 'CORRUPT', 'SENSOR_DEAD', 'MAINTENANCE']
Stasiun ppm: ['BJB-01_Banjarbaru_AQMS.csv', 'MDN-01_Medan_AQMS.csv', 'PSW-01_Pesawaran_AQMS.csv']


,median_stasiun_ppm,median_stasiun_ug,rasio_ug_per_ppm,faktor_fisik_gas
pm25_ug_m3,0.045,28.100,624.444,NaN
pm10_ug_m3,0.065,40.350,616.972,NaN
so2_ug_m3,0.003,8.400,"2,800.000","2,620.450"
co_ug_m3,0.456,384.400,842.982,"1,145.603"
o3_ug_m3,0.009,14.100,"1,658.824","1,963.190"
no2_ug_m3,0.015,19.600,"1,315.436","1,881.800"


Jika rasio ≈ faktor_fisik_gas → nilainya memang ppm (konversi benar). Jika rasio ≈ 1 → label ppm keliru.
PM2.5/PM10 tidak punya faktor fisik: lihat rasio untuk memutuskan PM_PPM_FACTOR di bagian 3.2.


### 2.5 Data kesehatan

In [32]:
hnum = health[HEALTH_COUNTS + HEALTH_WEATHER].apply(pd.to_numeric, errors="coerce")
h_date, h_fmt, h_fail = parse_ts(health["date"])
print("Format tanggal terdeteksi:", h_fmt, "| gagal baca:", h_fail, "%")
print("Rentang tanggal:", h_date.min(), "→", h_date.max())
print("Nilai kosong per kolom:"); display(health.isna().sum()[lambda s: s > 0].to_frame("kosong"))
print("Duplikat hospital_id+date+time:", int(health.duplicated(["hospital_id", "date", "time"]).sum()))
print("hospital_id tidak dikenal:", sorted(set(health["hospital_id"].dropna()) - set(hospitals["hospital_id"]))[:10])
print("Nilai teks di kolom angka:")
display(text_report(health.assign(source_file="health"), HEALTH_COUNTS + HEALTH_WEATHER))
display(hnum.describe(percentiles=[.01, .5, .99]).T)
print("Jumlah negatif per kolom:", (hnum[HEALTH_COUNTS] < 0).sum().to_dict())
print("Nilai sentinel per kolom:", {c: int(hnum[c].isin(SENTINELS).sum()) for c in hnum.columns if hnum[c].isin(SENTINELS).any()})

Format tanggal terdeteksi: %Y-%m-%d | gagal baca: 0.0 %
Rentang tanggal: 2023-12-31 00:00:00 → 2026-10-14 00:00:00
Nilai kosong per kolom:


,kosong


Duplikat hospital_id+date+time: 0
hospital_id tidak dikenal: []
Nilai teks di kolom angka:


""


,count,mean,std,min,1%,50%,99%,max
resp_admissions_under5,"27,491.000",6.773,5.406,0.000,1.000,5.000,25.000,114.000
resp_admissions_over60,"27,491.000",8.190,6.325,0.000,1.000,6.000,29.000,127.000
asthma_emergency_cases,"27,491.000",4.832,3.797,0.000,0.000,4.000,18.000,64.000
copd_admissions,"27,491.000",3.978,3.187,0.000,0.000,3.000,16.000,76.000
resp,"27,491.000",26.674,17.809,0.000,3.000,21.000,77.000,307.000
cvd,"27,491.000",7.519,3.856,0.000,1.000,7.000,17.000,61.000
death,"27,491.000",1.987,1.580,0.000,0.000,2.000,7.000,24.000
temp_c,"27,491.000",26.332,1.386,22.000,22.900,25.900,29.200,32.100
dptp_c,"27,491.000",22.336,1.236,18.100,20.000,21.600,25.300,29.300
rhum_pct,"27,491.000",79.759,3.310,64.900,74.900,78.600,87.800,99.000


Jumlah negatif per kolom: {'resp_admissions_under5': 0, 'resp_admissions_over60': 0, 'asthma_emergency_cases': 0, 'copd_admissions': 0, 'resp': 0, 'cvd': 0, 'death': 0}
Nilai sentinel per kolom: {}


## 3. Cleaning — bersihkan data
### 3.1 AQMS

In [33]:
aqms = aqms_raw.copy()
log("AQMS awal", len(aqms))

# (1) Teks error (0xDEADBEEF, ERR, #VALUE! ...) → NaN, ubah ke angka
for c in NUM_COLS:
    conv = pd.to_numeric(aqms[c], errors="coerce")
    n_text = int((conv.isna() & aqms[c].notna()).sum())
    aqms[c] = conv
    if n_text: log(f"{c}: teks error → NaN", n_text)

# (2) Kode penanda (-99.9, -999, 9999 ...) → NaN  [dilakukan SEBELUM konversi satuan]
for c in NUM_COLS:
    m = aqms[c].isin(SENTINELS)
    if m.any():
        aqms.loc[m, c] = np.nan
        log(f"{c}: kode penanda → NaN", m.sum())

[LOG] AQMS awal: 667,981 
[LOG] pm25_ug_m3: teks error → NaN: 6,290 
[LOG] pm10_ug_m3: teks error → NaN: 6,283 
[LOG] so2_ug_m3: teks error → NaN: 6,286 
[LOG] co_ug_m3: teks error → NaN: 6,407 
[LOG] o3_ug_m3: teks error → NaN: 6,321 
[LOG] no2_ug_m3: teks error → NaN: 6,279 
[LOG] temperature_c: teks error → NaN: 124 
[LOG] humidity_pct: teks error → NaN: 149 
[LOG] pm25_ug_m3: kode penanda → NaN: 891 
[LOG] pm10_ug_m3: kode penanda → NaN: 889 
[LOG] so2_ug_m3: kode penanda → NaN: 976 
[LOG] co_ug_m3: kode penanda → NaN: 7,571 
[LOG] o3_ug_m3: kode penanda → NaN: 5,835 
[LOG] no2_ug_m3: kode penanda → NaN: 967 
[LOG] temperature_c: kode penanda → NaN: 8,474 
[LOG] wind_speed_ms: kode penanda → NaN: 8,474 
[LOG] wind_direction_deg: kode penanda → NaN: 15,597 


In [34]:
# (3) Satuan ppm → µg/m³ (stasiun dengan unit_raw == "ppm")
# Gas: µg/m³ = ppm × BM × 1000 / 24,45  (25°C, 1 atm).
# PM2.5/PM10 tidak punya konversi fisik dari ppm. ⚙️ SESUAIKAN PM_PPM_FACTOR setelah melihat tabel "scale" (bagian 2.4f):
#   None  → nilai PM di stasiun ppm dikosongkan (aman, tapi PM stasiun itu hilang)
#   angka → dikalikan faktor tersebut (mis. 1.0 bila ternyata label ppm hanya keliru)
PM_PPM_FACTOR = None

is_ppm = aqms["unit_raw"].eq("ppm")
for c, mw in MW.items():
    aqms.loc[is_ppm, c] = aqms.loc[is_ppm, c] * mw * 1000 / 24.45
    log(f"{c}: konversi ppm → µg/m³", aqms.loc[is_ppm, c].notna().sum(), "(rumus fisik gas)")
for c in ["pm25_ug_m3", "pm10_ug_m3"]:
    n_pm = int(aqms.loc[is_ppm, c].notna().sum())
    if PM_PPM_FACTOR is None:
        aqms.loc[is_ppm, c] = np.nan
        log(f"{c}: nilai di stasiun ppm dikosongkan (faktor belum ditentukan)", n_pm)
    else:
        aqms.loc[is_ppm, c] = aqms.loc[is_ppm, c] * PM_PPM_FACTOR
        log(f"{c}: dikalikan faktor {PM_PPM_FACTOR}", n_pm)

# (4) Di luar batas fisik → NaN
for c, (lo, hi) in LIMITS.items():
    m = aqms[c].notna() & ((aqms[c] < lo) | (aqms[c] > hi))
    if m.any():
        aqms.loc[m, c] = np.nan
        log(f"{c}: di luar [{lo}, {hi}] → NaN", m.sum())

# (5) PM2.5 > PM10: hanya ditandai, tidak dihapus
aqms["flag_pm25_gt_pm10"] = aqms["pm25_ug_m3"] > aqms["pm10_ug_m3"] * 1.05
log("ditandai PM2.5 > PM10 (toleransi 5%)", aqms["flag_pm25_gt_pm10"].sum(), "tidak dihapus")

[LOG] so2_ug_m3: konversi ppm → µg/m³: 69,040 (rumus fisik gas)
[LOG] no2_ug_m3: konversi ppm → µg/m³: 69,040 (rumus fisik gas)
[LOG] co_ug_m3: konversi ppm → µg/m³: 69,040 (rumus fisik gas)
[LOG] o3_ug_m3: konversi ppm → µg/m³: 69,040 (rumus fisik gas)
[LOG] pm25_ug_m3: nilai di stasiun ppm dikosongkan (faktor belum ditentukan): 69,040 
[LOG] pm10_ug_m3: nilai di stasiun ppm dikosongkan (faktor belum ditentukan): 69,040 
[LOG] pm25_ug_m3: di luar [0, 1000] → NaN: 14,846 
[LOG] pm10_ug_m3: di luar [0, 2000] → NaN: 7,567 
[LOG] so2_ug_m3: di luar [0, 2000] → NaN: 14,202 
[LOG] co_ug_m3: di luar [0, 100000] → NaN: 799 
[LOG] o3_ug_m3: di luar [0, 1000] → NaN: 7,521 
[LOG] no2_ug_m3: di luar [0, 1000] → NaN: 12,425 
[LOG] temperature_c: di luar [-10, 50] → NaN: 12,743 
[LOG] humidity_pct: di luar [0, 100] → NaN: 18,255 
[LOG] wind_speed_ms: di luar [0, 60] → NaN: 12,024 
[LOG] wind_direction_deg: di luar [0, 360] → NaN: 8,619 
[LOG] ditandai PM2.5 > PM10 (toleransi 5%): 2,813 tidak dihapu

In [35]:
# (6) Filter berdasar qc_flag. ⚙️ SESUAIKAN QC_KEEP:
#   ["VALID", "SUSPECT"] → SUSPECT dipertahankan tapi ditandai (is_suspect). Ganti ["VALID"] untuk analisis paling ketat.
QC_KEEP = ["VALID", "SUSPECT"]
aqms["qc_flag"] = aqms["qc_flag"].astype("string").str.strip().str.upper()
bad_qc = ~aqms["qc_flag"].isin(QC_KEEP)
n_vals = int(aqms.loc[bad_qc, NUM_COLS].notna().sum().sum())
aqms.loc[bad_qc, NUM_COLS] = np.nan
log("qc_flag di luar QC_KEEP → seluruh nilai ukur NaN", n_vals, f"({int(bad_qc.sum()):,} baris, keep={QC_KEEP})")
aqms["is_suspect"] = aqms["qc_flag"].eq("SUSPECT")

# (7) Timestamp: buang yang rusak / di luar periode (mis. 1970-01-01), bulatkan ke jam
aqms["jam"] = aqms["ts_start"].dt.round("h")
bad_ts = aqms["jam"].isna() | (aqms["jam"] < START) | (aqms["jam"] > END)
log("hapus baris timestamp rusak / di luar periode", bad_ts.sum())
aqms = aqms[~bad_ts]

# (8) Duplikat & data sub-jam → satu baris per sensor per jam (rata-rata nilai valid)
agg = {c: "mean" for c in NUM_COLS}
agg.update({"flag_pm25_gt_pm10": "max", "is_suspect": "max", "measurement_id": "size", "unit_raw": "first"})
hourly = (aqms.groupby(["sensor_id", "jam"], as_index=False).agg(agg)
          .rename(columns={"measurement_id": "n_records"}))
log("baris digabung jadi 1 per sensor-jam (duplikat / sub-jam)", len(aqms) - len(hourly))
log("AQMS per jam (hasil)", len(hourly))

[LOG] qc_flag di luar QC_KEEP → seluruh nilai ukur NaN: 133,339 (49,055 baris, keep=['VALID', 'SUSPECT'])
[LOG] hapus baris timestamp rusak / di luar periode: 132 
[LOG] baris digabung jadi 1 per sensor-jam (duplikat / sub-jam): 12,041 
[LOG] AQMS per jam (hasil): 655,808 


In [36]:
# Simpan hasil cleaning AQMS

import os

# Membuat folder jika belum tersedia
OUTPUT_DIR = "../data/clean/aqms"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Menyimpan hasil cleaning
output_path = os.path.join(
    OUTPUT_DIR,
    "aqms_hourly_clean.csv"
)

hourly.to_csv(output_path, index=False)

print("Data cleaning berhasil disimpan!")
print("Lokasi:", output_path)
print("Jumlah baris:", len(hourly))
print("Jumlah kolom:", len(hourly.columns))

Data cleaning berhasil disimpan!
Lokasi: ../data/clean/aqms\aqms_hourly_clean.csv
Jumlah baris: 655808
Jumlah kolom: 16
